In [5]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
import scipy as sc

***Project 1***

In this project, the task was to create a weighted local simple regression model and apply it to a data set provided in the course BERN02. The WLSR is calculated by minimising
$\sum_{i=1}^{k}K_{i0}(y_i-\beta_0-\beta_1 x_i)^2$,
where $K_{i0}$ is the weight assigned at each point, $x_0$ is the initial grid point and $k$ is the number of nearest points to $x_0$. Then the fitted value at $x_0$ can be found by
$\hat{y} | x_0=\hat{f} (x_0)=\beta_0+\beta_1 x_0$.
The WLSR function takes four arguments: the response variable $y$, the predictor variable $x$, the number of neighbours $k$, and the point of interest $x_0$. It returns the predicted value and standard error at $x_0$.

In [6]:
def WLSR(y, x, k, x_0):
    '''This is a weighted least squares regression function that takes in 
    the response variable y, the predictor variable x, the number of 
    neighbours k, and the point of interest x_0. It returns the predicted 
    value and standard error at x_0.'''
    D = [] 
    for i in range(len(x)):
        # finding the distance between x_0 and each x value
        dist = np.abs(x[i] - x_0)
        D.append((dist))
    
    # sorting it by distance and taking the k nearest neighbours
    neigh = sorted(D)[:k]
    d_max = max(neigh)

    for j in range(len(D)):
        # normalizing the distance by the maximum distance
        # and applying the tricube weight function
        D[j] = D[j] / d_max
        if D[j] >= 1:
            D[j] = 0
        else:
            D[j] = (1 - D[j] ** 3) ** 3
    
    # adding a constant to the predictor variable for the intercept
    x = sm.add_constant(x)

    # fitting the weighted least squares regression model
    model = sm.WLS(y, x, weights=D)
    results = model.fit()

    # predicting the value at x_0
    int_pred = [1, x_0]

    pred_info = results.get_prediction(int_pred)

    # separating the predicted value and standard error from the prediction results
    pred = pred_info.predicted_mean[0]
    se = pred_info.se_mean[0]

    return pred, se

Then it returns the predicted values and standard errors. These results are then presented in a table as follows.

In [7]:
# reading in the data
cleandata = pd.read_csv("Data/pollution_cleaneddata.csv")

# separating the predictor and response variables
x_val = cleandata['POOR'].values
y_val = cleandata['MORT'].values

percent = [10, 18, 25]

# using the function
get_results = [WLSR(y_val, x_val, k = 20, x_0 = p) for p in percent]

# separating the predicted values and standard errors
pred = [result[0] for result in get_results]
se = [result[1] for result in get_results]

# creating a dataframe to display the results
df_out = pd.DataFrame({"POOR (%)": percent, "Predicted MORT": pred, "SE": se})

print(df_out)

   POOR (%)  Predicted MORT         SE
0        10      900.607323   8.091129
1        18      956.767778   6.747982
2        25     1010.030441  10.567187


***Project 2***

In this Project, the task was to formulate a model for Poisson regression with year as a predictor $Y|x_i\sim Po(\lambda(x_i))$. To be able to optimise the maximum likelihood estimator, the negative log likelihood function for poisson was used as follows. $NLL = \sum_{i=1}^{n}\exp(\beta_0 + \beta_1 * x_i)-y_i(\beta_0 + \beta_1 * x_i)+\log(y_i!)$.

In [8]:
def nll_Poisson(params, x, y):
    '''The negative log likelihood function for poisson 
    distribution, that takes three arguments, params with is an array
    of two elements, x and y which are the independent and 
    dependent variables respectively, and returns the negative 
    log likelihood value.'''
    beta0, beta1 = params #unpacking the parameters

    #calculating the negative log likelihood value
    term_1 = y * (beta0 + beta1 * x)
    term_2 = np.exp(beta0 + beta1 * x)
    term_3 = np.log(sc.special.factorial(y))
    nll = np.sum(- term_1 + term_2 + term_3)

    return nll

Then use the model with the point estimates of the parameters to generate a three samples of data given from the time period 1999 to 2012. Lastly, we save it as a csv data file.

In [9]:
#reading in the data and separating the independent and dependent variables
cleandata = pd.read_csv("Data/bird_count.csv")
years = cleandata["yr"].values
y_val = cleandata["count"].values
#adjusting the years to start from 0
x_val = years - years[0]

#initial guess for the parameters of the poisson distribution
params = [0.0, 0.0]

#fitting the poisson regression model using maximum likelihood estimation
max_likelihood = sc.optimize.minimize(nll_Poisson, params, args=(x_val, y_val))

#unpacking the estimated parameters from the optimization result
beta0_hat, beta1_hat = max_likelihood.x

#calculating the predicted values of the response variable
y_hat = np.exp(beta0_hat + beta1_hat * x_val)

#generating three random samples from the poisson distribution 
#with the predicted values as the mean
sample_1 = np.random.poisson(y_hat)
sample_2 = np.random.poisson(y_hat)
sample_3 = np.random.poisson(y_hat)

#saving the generated samples to a new csv file
df1 = pd.DataFrame({"yr": years, "sample": 1, "count": sample_1})
df2 = pd.DataFrame({"yr": years, "sample": 2, "count": sample_2})
df3 = pd.DataFrame({"yr": years, "sample": 3, "count": sample_3})

df = pd.concat([df1, df2, df3], ignore_index=True)

df.to_csv("bird_count_hw.csv", index=False)

***FAIR Principles***

**Findable:** The code and results is readily avaliable on GitHub, however only to those invited to collaborate. Also, the data provided in the course is not, because I am not sure if that is legal.

**Accessible:** The code should be accessible.

**Interoperable:** The code uses well known libraries and the results are presented in csv format.

**Reusable:** The resulting data can be reused, but again I am unsure if I can add the data provided in the course.

I can conclude that this specific repository does not fullfil the FAIR requirements, however it could if required.